# Prefix programs: figures and tables

Everything here is computed from the synced result tables (`results_sync/*.csv`, per-example JSONs and training histories) and `prereg.json`. No GPU, no model. Run top to bottom; figures land in `figures/`, tables in `tables/`.

Operating points are read from the preregistration and asserted; the 0.05 level and everything in sections marked *exploratory* were not preregistered.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from prefixlab.analysis import load, curves, frontier, distance, mechanism, compose, regime2, bootstrap, plots
RESULTS = ROOT / "results_sync"; FIG = ROOT / "figures"; TAB = ROOT / "tables"; FIG.mkdir(exist_ok=True); TAB.mkdir(exist_ok=True)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40); pd.set_option("display.precision", 3)
prereg = load.load_prereg(ROOT / "prereg.json")
OPS = tuple(prereg["operating_points_D_hat"]); assert OPS == (0.25, 0.10), "operating points are preregistered"
OPS_X = OPS + (0.05,)   # 0.05 is exploratory
T = load.load_tables(RESULTS)
{k: len(v) for k, v in T.items()}

{'sweep': 273,
 'flat': 72,
 'flat2': 20,
 'hier_task': 72,
 'hier_lang': 72,
 'compose': 72,
 'distance': 72,
 'regime2': 32,
 'mechanism': 1120,
 'recall_long': 1}

## 0. Scales: no-prefix loss and floor per cell

Floor = min(LoRA floor, best Design A prefix) on the cell's own scale (ledger ruling, a deviation from the preregistered LoRA-only floor). Continuation cells have a floor within 0.15 bits/token of the no-prefix model, so their D̂ is noise; they are kept in the tables but excluded from headline claims.

In [2]:
fl = load.floors(T["sweep"]); fl["gap"] = fl.L_none - fl.L_floor
fl.to_csv(TAB / "floors.csv", index=False); fl

,cell,L_none,L_lora,L_bestA,L_floor,gap
0,continuation-de,3.500,3.442,3.387,3.387,0.114
1,continuation-en,3.862,3.750,3.706,3.706,0.156
2,continuation-ru,2.923,2.871,2.827,2.827,0.096
3,recall-de,7.621,1.721,1.964,1.721,5.900
4,recall-en,6.056,2.584,2.789,2.584,3.472
5,recall-ru,7.342,1.102,1.285,1.102,6.241
6,translation-de,2.441,1.560,1.537,1.537,0.903
7,translation-ru,1.924,1.353,1.343,1.343,0.580


## 1. Slots are not a description length (H1, Design A)

D̂ against k for every cell and conditioning; k* at the preregistered levels; seed gaps; the natural-language prompt against one trained slot.

In [3]:
sw = load.attach_D(T["sweep"], fl)
fig = plots.plot_curves(sw, ops=OPS); fig.savefig(FIG / "curves.png", dpi=150); plt.show()

/var/folders/7k/_xmp_c2n2kxb1_ygsnsb3ghr0000gn/T/ipykernel_80622/2596465381.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plots.plot_curves(sw, ops=OPS); fig.savefig(FIG / "curves.png", dpi=150); plt.show()


In [4]:
ks = curves.k_star_table(sw, OPS_X); ks.to_csv(TAB / "k_star.csv", index=False)
ks.pivot(index="cell", columns="cond")

k*(0.25)           k*(0.1)           k*(0.05)            
cond                lang none task    lang none task     lang  none  task
cell                                                                     
continuation-de        1    2    1       1    2    2      4.0   4.0   4.0
continuation-en        1    1    1       1    2    1      1.0   2.0   1.0
continuation-ru        1    2    1       1    2    1      2.0   2.0   8.0
recall-de              1    1    1       4    4    4     32.0  32.0  32.0
recall-en              2    2    2       8   16    8      NaN   NaN   NaN
recall-ru              1    1    1       1    2    1      8.0   4.0   4.0
translation-de         1    1    1       1    1    1      2.0   2.0   1.0
translation-ru         1    1    1       1    1    1      2.0   1.0   1.0

In [5]:
nl = curves.nl_prompt_table(sw); nl.to_csv(TAB / "nl_prompt.csv", index=False)
gaps = curves.seed_gap_table(sw); gaps.to_csv(TAB / "seed_gaps.csv", index=False)
display(nl); gaps

,cell,D_nl_prompt,D_one_slot,D_nl_prompt_plus_one_slot
0,continuation-de,0.953,0.267,NaN
1,continuation-en,0.714,0.212,NaN
2,continuation-ru,1.168,0.271,NaN
3,recall-de,0.930,0.224,0.153
4,recall-en,0.958,0.407,NaN
5,recall-ru,0.910,0.128,0.090
6,translation-de,0.125,0.051,0.025
7,translation-ru,0.197,0.047,0.023


,cell,max_seed_gap
0,continuation-de,0.102
1,continuation-en,0.083
2,continuation-ru,0.047
3,recall-de,0.006
4,recall-en,0.196
5,recall-ru,0.042
6,translation-de,0.007
7,translation-ru,0.011


### H2 on the slot axis: Δk from conditioning
k*(none) − k*(lang) and k*(none) − k*(task) per cell at D̂ ≤ 0.10. Mostly zero: a conditioning string is worth less than one slot.

In [6]:
sav = curves.savings_table(ks, 0.10); sav.to_csv(TAB / "savings_k.csv", index=False); sav

,cell,k_none,k_lang,k_task,dk_lang,dk_task
0,continuation-de,2,1,2,1,0
1,continuation-en,2,1,1,1,1
2,continuation-ru,2,1,1,1,1
3,recall-de,4,4,4,0,0
4,recall-en,16,8,8,8,8
5,recall-ru,2,1,1,1,1
6,translation-de,1,1,1,0,0
7,translation-ru,1,1,1,0,0


### Recall is budget-bound (exploratory check)
recall-ru, k = 8, 3000 steps against the 1500-step sweep runs.

In [7]:
rl = load.attach_D(T["recall_long"], fl)
cols = ["cell", "k", "seed", "steps", "best_step", "test_bits_per_token", "D"]
pd.concat([sw[(sw.design == "A") & (sw.cell == "recall-ru") & (sw.k == 8) & (sw.cond == "none")][cols], rl[cols]]) if len(rl) else "no recall_long table" 

,cell,k,seed,steps,best_step,test_bits_per_token,D
36,recall-ru,8,0,1500,1400,1.368,0.043
37,recall-ru,8,1,1500,1500,1.366,0.042
0,recall-ru,8,0,3000,2900,1.301,0.032


## 2. Description length in bits: the variational frontier

Rate = KL of the diagonal-Gaussian posterior to the embedding prior (bits-back). Distortion is scored on posterior **samples**, because the sample is the coded object; the posterior mean is not a program the model was trained on (compare the two columns at β = 3).

Two protocols: `flat` (preregistered: lr 0.05, checkpoint = lowest mean-dev distortion, σ initialised at 5× the prior) and `flat2` (lr 0.01, checkpoint = lowest dev objective, dev scored with samples, σ at the prior). `flat2` is the headline where it exists; the protocol change is recorded as a deviation.

In [8]:
f1 = load.attach_D(load.attach_D(T["flat"], fl, "test_bits_per_token_sampled", "D_sampled"), fl)
f2 = load.attach_D(load.attach_D(T["flat2"], fl, "test_bits_per_token_sampled", "D_sampled"), fl)
fr1 = frontier.frontier(f1, OPS_X, cells=load.MEASURABLE).assign(protocol="flat")
fr2 = frontier.frontier(f2, OPS_X, cells=load.MEASURABLE).assign(protocol="flat2")
front = pd.concat([fr2, fr1]); front.to_csv(TAB / "frontier.csv", index=False)
front.pivot(index="cell", columns=["protocol", "op"], values="bits").round(0)

protocol         flat2                    flat                
op                0.25    0.10   0.05     0.25    0.10    0.05
cell                                                          
recall-de       3343.0     NaN    NaN   4168.0     NaN     NaN
recall-en       7155.0     NaN    NaN  24686.0     NaN     NaN
recall-ru       2419.0  2419.0    NaN   3101.0  3101.0     NaN
translation-de   436.0   436.0  952.0   1194.0  1194.0  2292.0
translation-ru   542.0   542.0  861.0    699.0   699.0  1210.0

In [9]:
fig = plots.plot_frontier({"flat": f1, "flat2": f2}, load.MEASURABLE, ops=OPS); fig.savefig(FIG / "frontier.png", dpi=150); plt.show()

/var/folders/7k/_xmp_c2n2kxb1_ygsnsb3ghr0000gn/T/ipykernel_80622/2028876795.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plots.plot_frontier({"flat": f1, "flat2": f2}, load.MEASURABLE, ops=OPS); fig.savefig(FIG / "frontier.png", dpi=150); plt.show()


In [10]:
# posterior mean vs sampled distortion, flat2 rows
f2[["cell", "k", "beta", "kl_bits", "best_step", "D", "D_sampled"]].rename(columns={"D": "D_mean"}).sort_values(["cell", "k", "beta"])

,cell,k,beta,kl_bits,best_step,D_mean,D_sampled
16,recall-de,2,1.0,8686.333,1400,0.256,0.252
12,recall-de,2,3.0,3715.545,1500,0.281,0.277
17,recall-de,4,1.0,8616.862,1400,0.134,0.143
13,recall-de,4,3.0,3342.839,1500,0.169,0.168
18,recall-en,2,1.0,5933.449,1400,0.247,0.253
14,recall-en,2,3.0,2352.445,1400,0.257,0.283
19,recall-en,4,1.0,7155.348,1300,0.203,0.226
15,recall-en,4,3.0,3158.767,1400,0.295,0.313
8,recall-ru,2,1.0,5072.724,1500,0.085,0.083
9,recall-ru,2,3.0,2418.629,1400,0.088,0.089


### Optimisation: overshoot and selection (from the saved histories)
KL and dev loss against step for a k = 2 and a k = 8 hierarchical run and a flat2 run. With lr 0.05 the KL overshoots by a factor that grows with k and the distortion-only rule selects the high-rate step.

In [11]:
def hist(table, run_prefix):
    r = T[table][T[table].run_id.str.startswith(run_prefix)]
    return frontier.dev_trajectory(load.load_history(RESULTS, r.history.iloc[0])) if len(r) and "history" in r else pd.DataFrame()
trajs = {"hier translation k=2 β=1 (lr 0.05)": hist("hier_task", "hier_task-translation_none_k2_s0_b1_t1p"),
         "hier translation k=8 β=1 (lr 0.05)": hist("hier_task", "hier_task-translation_none_k8_s0_b1_t1p"),
         "flat2 translation-ru k=2 β=1 (lr 0.01)": hist("flat2", "flat2_translation-ru_none_k2_s0_b1")}
trajs = {k: v for k, v in trajs.items() if len(v)}
fig = plots.plot_trajectories(trajs); fig.savefig(FIG / "trajectories.png", dpi=150); plt.show()

/var/folders/7k/_xmp_c2n2kxb1_ygsnsb3ghr0000gn/T/ipykernel_80622/335785126.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plots.plot_trajectories(trajs); fig.savefig(FIG / "trajectories.png", dpi=150); plt.show()


## 3. Task versus language as the shared factor (H2 in bits, Design B)

One centre per task with language deviations, or one centre per language with task deviations; τ = 1 in units of the prior σ. Total bits over all eight cells at D̂ ≤ 0.25 under the three codes.

In [12]:
ht = load.attach_D(T["hier_task"], fl, "test_bits_per_token_sampled", "D_sampled")
hl = load.attach_D(T["hier_lang"], fl, "test_bits_per_token_sampled", "D_sampled")
tt = frontier.hier_totals(ht, OPS); tl = frontier.hier_totals(hl, OPS)
tt.to_csv(TAB / "hier_task_totals.csv", index=False); tl.to_csv(TAB / "hier_lang_totals.csv", index=False)
display(tt.round(0)); tl.round(0)

,group,op,k,beta,centre,total,n_members,dev_ru,dev_de,dev_en
0,task:continuation,0.0,2.0,0.0,2089.0,3568.0,3,443.0,560.0,476.0
1,task:continuation,0.0,4.0,0.0,3979.0,7195.0,3,1085.0,1296.0,835.0
2,task:recall,0.0,2.0,1.0,16531.0,27223.0,3,3318.0,4030.0,3343.0
3,task:recall,0.0,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
4,task:translation,0.0,2.0,1.0,1586.0,2733.0,2,485.0,662.0,NaN
5,task:translation,0.0,2.0,1.0,1586.0,2733.0,2,485.0,662.0,NaN


,group,op,k,beta,centre,total,n_members,dev_translation,dev_recall,dev_continuation
0,lang:de,0.0,4.0,3.0,1413.0,7283.0,3,438.0,5320.0,112.0
1,lang:de,0.0,NaN,NaN,NaN,NaN,3,NaN,NaN,NaN
2,lang:en,0.0,4.0,0.0,8329.0,35227.0,2,NaN,26157.0,741.0
3,lang:en,0.0,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN
4,lang:ru,0.0,2.0,1.0,2954.0,10966.0,3,632.0,7204.0,176.0
5,lang:ru,0.0,4.0,0.0,12759.0,38147.0,3,1917.0,22729.0,742.0


In [13]:
fr1_all = frontier.frontier(f1, OPS, cells=load.CELLS)   # flat covers all 8 cells (flat2 only the measurable ones)
cmp = frontier.scheme_comparison(fr1_all, tt, tl, 0.25, load.CELLS); cmp.to_csv(TAB / "scheme_comparison.csv", index=False); cmp

,scheme,bits,complete
0,task-centred hierarchy,33524.551,True
1,independent flat programs,37571.343,True
2,language-centred hierarchy,53475.787,True


In [14]:
fig = plots.plot_hier_bars(tt, tl, 0.25); fig.savefig(FIG / "hier_bars.png", dpi=150); plt.show()

/var/folders/7k/_xmp_c2n2kxb1_ygsnsb3ghr0000gn/T/ipykernel_80622/3288151964.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plots.plot_hier_bars(tt, tl, 0.25); fig.savefig(FIG / "hier_bars.png", dpi=150); plt.show()


Per-member view: what a language costs given the task centre, and what a task costs given the language centre (cheapest run reaching D̂ ≤ 0.25 per group).

In [15]:
def member_bits(h, totals, op):
    rows = []
    for r in totals[totals.op == op].itertuples():
        g = h[(h.group == r.group) & (h.k == r.k) & (h.beta == r.beta)]
        for m in g.itertuples():
            rows.append(dict(group=r.group, member=m.member, cell=m.cell, centre=round(r.centre), deviation=round(m.kl_dev_bits), D=m.D_sampled))
    return pd.DataFrame(rows)
mb = pd.concat([member_bits(ht, tt, 0.25), member_bits(hl, tl, 0.25)]); mb.to_csv(TAB / "hier_members.csv", index=False); mb

,group,member,cell,centre,deviation,D
0,task:continuation,ru,continuation-ru,2089,443,0.093
1,task:continuation,de,continuation-de,2089,560,0.120
2,task:continuation,en,continuation-en,2089,476,0.074
3,task:recall,ru,recall-ru,16531,3318,0.085
4,task:recall,de,recall-de,16531,4030,0.167
5,task:recall,en,recall-en,16531,3343,0.242
6,task:translation,ru,translation-ru,1586,485,0.045
7,task:translation,de,translation-de,1586,662,0.052
0,lang:de,translation,translation-de,1413,438,0.091
1,lang:de,recall,recall-de,1413,5320,0.200


## 4. Composition (H3): tied task parts + language parts

Six parts for eight cells. Trained on every cell, and assembled zero-shot for held-out cells.

In [16]:
c = load.attach_D(T["compose"], fl)
ho = compose.holdout_table(c, sw); ho.to_csv(TAB / "compose_holdout.csv", index=False)
svf = compose.shared_vs_flat(c, sw); svf.to_csv(TAB / "compose_shared_vs_flat.csv", index=False)
display(ho); svf.pivot(index="cell", columns=["k_task"], values=["D_composed", "D_designA_same_k"])

,cell,k_task,k_lang,holdout,D_assembled,D_trained,D_designA,gap_closed
0,recall-de,1,1,continuation-ru+recall-de,0.741,0.172,0.084,0.259
1,continuation-ru,1,1,continuation-ru+recall-de,0.834,0.135,0.043,0.166
2,recall-de,2,2,continuation-ru+recall-de,0.675,0.157,0.084,0.325
3,continuation-ru,2,2,continuation-ru+recall-de,0.201,0.043,0.043,0.799
4,recall-de,4,4,continuation-ru+recall-de,0.677,0.133,0.084,0.323
5,continuation-ru,4,4,continuation-ru+recall-de,1.102,-0.024,0.043,-0.102
6,translation-de,2,2,recall-ru+translation-de,0.397,0.027,0.007,0.603
7,recall-ru,2,2,recall-ru+translation-de,0.858,0.076,0.049,0.142
8,translation-ru,2,2,recall-en+translation-ru,0.265,0.017,0.005,0.735
9,recall-en,2,2,recall-en+translation-ru,0.775,0.250,0.118,0.225


D_composed               D_designA_same_k              
k_task                   1      2      4                1      2      4
cell                                                                   
continuation-de      0.122  0.102  0.035            0.086  0.031  0.078
continuation-en      0.186  0.063  0.025            0.043  0.019  0.006
continuation-ru      0.135  0.043 -0.024            0.032  0.043  0.089
recall-de            0.172  0.157  0.133            0.106  0.084  0.068
recall-en            0.241  0.250  0.217            0.248  0.118  0.144
recall-ru            0.177  0.076  0.072            0.057  0.049  0.043
translation-de       0.040  0.027  0.029            0.011  0.007  0.005
translation-ru       0.034  0.017  0.022            0.007  0.005  0.002

In [17]:
fig = plots.plot_compose(ho[ho.cell.isin(load.MEASURABLE)]); fig.savefig(FIG / "compose.png", dpi=150); plt.show()

/var/folders/7k/_xmp_c2n2kxb1_ygsnsb3ghr0000gn/T/ipykernel_80622/43327706.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plots.plot_compose(ho[ho.cell.isin(load.MEASURABLE)]); fig.savefig(FIG / "compose.png", dpi=150); plt.show()


## 5. Distance between programs (H4)

Warm-start prequential learning cost from every cell's k = 4 prefix (and from scratch) toward every cell, 512-example budget. Symmetrised and divided by the from-scratch costs; silhouette of the task partition versus the language partition.

In [18]:
M, scratch = distance.cost_matrix(T["distance"], cells=load.CELLS)
S = distance.symmetrised_distance(M, scratch); S.to_csv(TAB / "distance_symmetrised.csv")
rel = distance.relative_cost(M, scratch); rel.to_csv(TAB / "distance_relative.csv")
gm = distance.pair_group_means(S); sil = distance.clustering_silhouettes(S)
gm.to_csv(TAB / "distance_groups.csv", index=False); sil.to_csv(TAB / "distance_silhouette.csv", index=False)
display(gm); sil

,pairs,mean,n,sd
0,same task,0.852,7,0.193
1,same language,0.952,7,0.218
2,neither,0.946,14,0.137


,partition,silhouette
0,task,0.004
1,language,-0.095


In [19]:
Z = distance.zero_shot_matrix(T["distance"], fl, cells=load.CELLS); Z.to_csv(TAB / "zero_shot.csv")
fig = plots.plot_distance(S, Z); fig.savefig(FIG / "distance.png", dpi=150); plt.show()
Z.round(2)

/var/folders/7k/_xmp_c2n2kxb1_ygsnsb3ghr0000gn/T/ipykernel_80622/2756045058.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plots.plot_distance(S, Z); fig.savefig(FIG / "distance.png", dpi=150); plt.show()


dst,translation-ru,translation-de,recall-ru,recall-de,recall-en,continuation-ru,continuation-de,continuation-en
src,,,,,,,,
translation-ru,0.02,1.22,0.71,0.97,0.94,0.88,3.34,1.81
translation-de,0.87,0.03,0.88,0.77,0.94,0.95,0.96,2.08
recall-ru,0.87,0.97,0.04,1.37,2.54,4.52,10.16,22.40
recall-de,1.35,0.87,0.90,0.08,1.36,19.31,5.92,25.42
recall-en,1.18,1.14,0.99,1.00,0.11,4.62,5.37,5.33
continuation-ru,0.89,0.93,0.97,1.02,0.86,-0.25,0.05,-0.16
continuation-de,0.88,0.55,1.00,0.80,0.88,0.55,-0.23,-0.00
continuation-en,1.02,1.01,1.02,1.01,0.84,0.26,0.65,-0.35


## 6. Where the program is read (H5): per-block patching

Recipient = random prefix, donor = trained prefix; the prefix positions' hidden states are patched into one block's attention input. `recovered / (recipient − donor)` per block. Probes: `A` (Design A k = 4), `hier_*_centre` (random → centre), `hier_*_dev` (centre → centre + deviation). Fractions are undefined when the donor is not at least 0.1 bits/token better than the recipient.

In [20]:
m = mechanism.recovery(T["mechanism"])
san = mechanism.sanity(m); assert (san.all_minus_donor.abs() < 1e-6).all(), "all-layers patch must reproduce the donor"
ds = mechanism.depth_summary(m); ds.to_csv(TAB / "mechanism_depth.csv", index=False); ds

,probe,task,n_cells,effect,peak_block,peak_frac,sum_frac,early,mid,late
0,A,continuation,3,0.196,0,0.814,3.042,1.501,0.875,0.666
1,A,recall,3,4.600,0,0.451,1.319,0.636,0.296,0.387
2,A,translation,2,0.584,0,0.787,2.555,1.218,0.684,0.654
3,hier_lang_centre,continuation,3,0.155,0,0.997,3.426,1.868,0.916,0.643
4,hier_lang_centre,recall,3,1.037,0,1.005,3.076,1.789,0.608,0.678
5,hier_lang_centre,translation,2,0.485,0,0.993,2.359,1.363,0.353,0.642
6,hier_lang_dev,recall,3,3.118,0,0.996,1.701,1.031,0.205,0.465
7,hier_task_centre,continuation,2,0.183,0,0.991,2.664,1.761,0.473,0.429
8,hier_task_centre,recall,2,3.643,0,0.902,1.796,0.996,0.336,0.464
9,hier_task_centre,translation,2,0.415,0,1.006,3.223,1.592,0.965,0.666


In [21]:
profiles = {p: mechanism.layer_profile(m, p) for p in ["A", "hier_task_centre", "hier_task_dev", "hier_lang_centre", "hier_lang_dev"]}
profiles = {p: v for p, v in profiles.items() if v.shape[1] > 0}
fig = plots.plot_mechanism(profiles); fig.savefig(FIG / "mechanism.png", dpi=150); plt.show()

/var/folders/7k/_xmp_c2n2kxb1_ygsnsb3ghr0000gn/T/ipykernel_80622/2340406263.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plots.plot_mechanism(profiles); fig.savefig(FIG / "mechanism.png", dpi=150); plt.show()


In [22]:
pk = mechanism.peak_blocks(m); pk.to_csv(TAB / "mechanism_peaks.csv"); pk

probe,A,hier_lang_centre,hier_lang_dev,hier_task_centre,hier_task_dev
cell,,,,,
continuation-de,0.0,0.0,NaN,0.0,NaN
continuation-en,0.0,0.0,NaN,0.0,NaN
continuation-ru,0.0,0.0,NaN,NaN,NaN
recall-de,0.0,0.0,0.0,0.0,0.0
recall-en,0.0,0.0,0.0,NaN,0.0
recall-ru,0.0,0.0,0.0,0.0,0.0
translation-de,0.0,0.0,NaN,0.0,0.0
translation-ru,0.0,0.0,NaN,0.0,NaN


## 7. Regime 2 (H6, exploratory): per-example input vectors

No rate term on the vectors, so this is a slot count. m* = smallest m whose residual loss is ≤ 10% of the loss with the real input, with (k = 4) and without (k = 0) the shared task prefix.

In [23]:
ms = regime2.m_star_table(T["regime2"]); ms.to_csv(TAB / "regime2_mstar.csv", index=False); ms

,cell,k,real_input,no_input,threshold,m_star,after_m1,after_m2,after_m4,after_m8
0,continuation-ru,0,2.911,3.707,0.291,2,5.639e-01,1.486e-01,3.096e-03,1.314e-03
1,continuation-ru,4,2.824,4.358,0.282,1,3.214e-02,5.213e-03,2.372e-03,1.207e-03
2,recall-ru,0,7.207,3.065,0.721,1,2.160e-03,2.907e-04,1.101e-04,4.681e-05
3,recall-ru,4,1.291,3.754,0.129,1,7.338e-04,3.399e-04,1.787e-04,9.032e-05
4,translation-de,0,2.535,4.312,0.254,2,1.442e+00,1.323e-01,8.079e-03,2.836e-03
5,translation-de,4,1.646,3.921,0.165,1,8.685e-02,9.720e-03,4.350e-03,1.899e-03
6,translation-ru,0,1.973,3.498,0.197,2,1.267e+00,1.384e-01,1.139e-02,3.194e-03
7,translation-ru,4,1.393,3.252,0.139,2,2.372e-01,2.132e-02,5.289e-03,1.927e-03


## 8. Recall across languages on common items (exploratory check)

The mLAMA test sets differ per language; the same comparison restricted to uuids present in all three.

In [24]:
try:
    from prefixlab.analysis import recall_common
    cu = recall_common.common_uuid_bits(RESULTS, ROOT / "data_raw" / "mlama", T["sweep"]); cu.to_csv(TAB / "recall_common_uuid.csv", index=False); display(cu)
except Exception as e:
    print("skipped:", e)

,cell,n_test,n_common,none,floor,prefix,D_prefix
0,recall-ru,4451,4212,7.324,1.085,1.392,0.049
1,recall-de,5534,4212,7.711,1.712,2.188,0.079
2,recall-en,6496,4212,5.656,2.386,2.786,0.122


## 9. Bootstrap intervals for the headline frontier points (over test examples)

In [25]:
rows = []
for r in fr2[fr2.op == 0.10].dropna(subset=["bits"]).itertuples():
    run = f2[(f2.cell == r.cell) & (f2.k == r.k) & (f2.beta == r.beta)].iloc[0]
    pe = load.load_per_example(RESULTS, run.per_example); base = T["sweep"][(T["sweep"].cell == r.cell) & (T["sweep"].design == "none") & (T["sweep"].cond == "none")].iloc[0]
    pe0 = load.load_per_example(RESULTS, base.per_example)
    if pe and pe0:
        pt, lo, hi = bootstrap.paired_ratio_diff_ci(pe["bits"], pe0["bits"], pe["tokens"])
        rows.append(dict(cell=r.cell, k=r.k, beta=r.beta, bits=round(r.bits), D_mean_eval=run.D, gain_bits_per_token=-pt, ci_lo=-hi, ci_hi=-lo))
ci = pd.DataFrame(rows); ci.to_csv(TAB / "frontier_ci.csv", index=False); ci

,cell,k,beta,bits,D_mean_eval,gain_bits_per_token,ci_lo,ci_hi
0,translation-ru,4.0,3.0,542,0.101,0.522,0.508,0.535
1,translation-de,2.0,3.0,436,0.441,0.505,0.471,0.539
2,recall-ru,2.0,3.0,2419,0.088,5.689,5.615,5.758
